In [ ]:
using LowLevelFEM

[ Info: Precompiling LowLevelFEM [6171b9fb-adbf-4751-adb9-5faded75de07] (cache misses: include_dependency fsize change (2), wrong dep version loaded (6), incompatible header (10))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


In [1]:
gmsh.initialize()

LoadError: UndefVarError: `gmsh` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
structured_rect_mesh(lx=2, order=2)

In [ ]:
material = Material("surface")

Pp = Problem([material], type=:ScalarField, field=:p, rhs_field=:fp, dim=2)
Pv = Problem([material], type=:VectorField, field=:v, rhs_field=:fv, dim=2)

In [ ]:
pres0 = BoundaryCondition("rightbottom", problem=Pp, p=0)

In [ ]:
suppT = BoundaryCondition("top", problem=Pv, vx=0, vy=0)
suppB = BoundaryCondition("bottom", problem=Pv, vx=0, vy=0)

In [ ]:
load_v = LoadCondition("surface", fvx=1.0, fvy=0.0)
fv = loadVector(Pv, [load_v])

gp = loadVector(Pp, [])

F = SystemVector([fv, gp])

In [ ]:
μ = 1.0

# stabilization
γ = 1e-1          # grad-div ( 1e-2...1e0)
δ = 1e-6          # pressure Laplacian (mesh dependent)

A = ∫((SymGrad(Pv) ⋅ SymGrad(Pv)) * 2μ)
D = ∫(Div(Pv) ⋅ Div(Pv) * γ)

# alternative way
AD = ∫((SymGrad(Pv) ⋅ SymGrad(Pv)) * 2μ + γ * (Div(Pv) ⋅ Div(Pv)))

B = ∫(Div(Pv) ⋅ Pp)
C = ∫(Grad(Pp) ⋅ Grad(Pp) * δ)

K = SystemMatrix([A+D B';
    B -C])

In [ ]:
K[:, :]

In [ ]:
v, p = solveField(K, F, support=[pres0, suppT, suppB])

In [ ]:
showDoFResults(v, name="v", visible=true)
showDoFResults(p, name="p")

openPostProcessor()

In [ ]:
gmsh.finalize()